# CarQeemat model development

This notebook documents the data cleaning, feature engineering, model training, and evaluation workflow for CarQeemat, a used-car asking-price estimator for Pakistan.

- Source listings: `data/raw/olx_cars.csv` (not included in the repository)
- Cleaned listings: 8,976 records, generated under `data/processed/`
- Deployed model: `models/carqeemat_pipeline.pkl`
- Test performance: MAE Rs 158,226; RMSE Rs 232,512; R² 0.9589

Run this notebook from the project root or its `notebooks/` directory. Add the source dataset locally before running the data-loading cells. Notebook outputs are intentionally cleared for repository hygiene; execute cells to regenerate them.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError("Could not find the CarQeemat project root.")

In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data/raw/olx_cars.csv")
print(df.shape)

In [ ]:
df.head()

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
df.duplicated().sum()

In [ ]:
df["Ad ID"].duplicated().sum()

In [ ]:
df[["Year", "KM's driven", "Price"]].describe()

In [ ]:
df["Make"].value_counts()

In [ ]:
df["Transmission"].value_counts()

In [ ]:
df[df["Ad ID"].duplicated(keep=False)].sort_values("Ad ID").head(20)

In [ ]:
df[df.duplicated(keep=False)].head(10)

In [ ]:
df.nlargest(10, "Price")[["Make", "Model", "Year", "KM's driven", "Price"]]

In [ ]:
df.nsmallest(10, "Price")[["Make", "Model", "Year", "KM's driven", "Price"]]

In [ ]:
df["Price"].value_counts().head(15)

In [ ]:
df["Ad ID"].value_counts().loc[lambda x: x > 1].shape[0]

In [ ]:
duplicate_ids = df[df["Ad ID"].duplicated(keep=False)]

differences = duplicate_ids.groupby("Ad ID").nunique()

conflicting_ids = differences[(differences > 1).any(axis=1)]

conflicting_ids

In [ ]:
df[df["Ad ID"].isin(conflicting_ids.index)].sort_values("Ad ID")

In [ ]:
df.duplicated().sum(), df.duplicated(keep=False).sum()

In [ ]:
for ad_id in conflicting_ids.index:
    display(df[df["Ad ID"] == ad_id][["Ad ID", "Registration city", "Price", "Car Profile"]])

In [ ]:
df_clean = df.drop_duplicates().copy()

In [ ]:
conflicting_ad_ids = [1080762189, 1080784426]

df_clean.loc[
    df_clean["Ad ID"].isin(conflicting_ad_ids),
    "Registration city"
] = pd.NA

In [ ]:
df_clean = df_clean.drop_duplicates(subset="Ad ID", keep="first").copy()

In [ ]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("Remaining exact duplicates:", df_clean.duplicated().sum())
print("Remaining duplicate Ad IDs:", df_clean["Ad ID"].duplicated().sum())

print("\nConflicting listings after cleaning:")
display(
    df_clean[df_clean["Ad ID"].isin(conflicting_ad_ids)][
        ["Ad ID", "Make", "Model", "Year", "Price", "Registration city"]
    ]
)

In [ ]:
display(
    df_clean[["Year", "KM's driven", "Price"]].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

In [ ]:
display(
    df_clean.nlargest(10, "Price")[
        ["Make", "Model", "Year", "KM's driven", "Price"]
    ]
)

display(
    df_clean.nsmallest(10, "Price")[
        ["Make", "Model", "Year", "KM's driven", "Price"]
    ]
)

In [ ]:
print("Cars priced at Rs. 5,000,000:")
print((df_clean["Price"] == 5000000).sum())

print("\nCars with mileage above 300,000 km:")
display(
    df_clean[df_clean["KM's driven"] > 300000][
        ["Make", "Model", "Year", "KM's driven", "Price"]
    ].sort_values("KM's driven", ascending=False)
)

In [ ]:
print("Cars priced at Rs. 4.5 million or above:")
print((df_clean["Price"] >= 4500000).sum())

print("\nCars with mileage below 1,000 km:")
display(
    df_clean[df_clean["KM's driven"] < 1000][
        ["Make", "Model", "Year", "KM's driven", "Price"]
    ].sort_values("KM's driven")
)

print("\nCars with exactly 500,000 km:")
print((df_clean["KM's driven"] == 500000).sum())

print("\nMost frequent prices:")
display(df_clean["Price"].value_counts().head(20))

print("\nMost frequent mileage values:")
display(df_clean["KM's driven"].value_counts().head(15))


In [ ]:
suspicious_mileage = [1, 20, 12345, 123456, 500000]

display(
    df_clean[df_clean["KM's driven"].isin(suspicious_mileage)][
        ["Ad ID", "Make", "Model", "Year", "KM's driven", "Price",
         "Registration city", "Seller Location"]
    ].sort_values("KM's driven")
)

In [ ]:
display(
    df_clean[df_clean["Price"] == 5000000][
        ["Make", "Model", "Year", "KM's driven", "Price",
         "Registration city"]
    ].sort_values(["Make", "Model", "Year"])
)

In [ ]:
spec_columns = ["Make", "Model", "Year", "KM's driven", "Price"]

possible_duplicates = df_clean[
    df_clean.duplicated(subset=spec_columns, keep=False)
].sort_values(spec_columns)

display(possible_duplicates[
    ["Ad ID", "Make", "Model", "Year", "KM's driven", "Price",
     "Registration city", "Seller Location"]
])

In [ ]:
spec_columns = ["Make", "Model", "Year", "KM's driven", "Price"]

possible_duplicates = df_clean[
    df_clean.duplicated(subset=spec_columns, keep=False)
].copy()

duplicate_groups = (
    possible_duplicates
    .groupby(spec_columns)
    .agg(
        listings=("Ad ID", "count"),
        unique_sellers=("Seller Location", "nunique"),
        unique_profiles=("Car Profile", "nunique"),
        unique_cities=("Registration city", "nunique")
    )
    .reset_index()
    .sort_values("listings", ascending=False)
)

display(duplicate_groups.head(20))

In [ ]:
print("Listings involved:", len(possible_duplicates))
print("Matching specification groups:", len(duplicate_groups))
print(
    "Groups with identical seller locations:",
    (duplicate_groups["unique_sellers"] == 1).sum()
)
print(
    "Groups with identical profile URLs:",
    (duplicate_groups["unique_profiles"] == 1).sum()
)

In [ ]:
same_location_groups = duplicate_groups[
    duplicate_groups["unique_sellers"] == 1
].sort_values("listings", ascending=False)

display(same_location_groups.head(10))

In [ ]:
sample_group = same_location_groups.iloc[0]

display(
    df_clean[
        (df_clean["Make"] == sample_group["Make"]) &
        (df_clean["Model"] == sample_group["Model"]) &
        (df_clean["Year"] == sample_group["Year"]) &
        (df_clean["KM's driven"] == sample_group["KM's driven"]) &
        (df_clean["Price"] == sample_group["Price"])
    ][
        ["Ad ID", "Car Name", "Make", "Model", "Year",
         "KM's driven", "Price", "Seller Location",
         "Description", "Car Profile"]
    ]
)

In [ ]:
profile_counts = (
    df_clean.groupby("Car Profile")["Ad ID"]
    .agg(["count", "nunique"])
    .reset_index()
)

repeated_profiles = profile_counts[
    (profile_counts["count"] > 1) &
    (profile_counts["Car Profile"].notna())
]

print("Repeated profile URLs:", len(repeated_profiles))

display(repeated_profiles.head(20))

In [ ]:
display(
    df_clean[
        df_clean["Car Profile"].isin(
            repeated_profiles["Car Profile"]
        )
    ][
        ["Ad ID", "Car Name", "Year", "KM's driven",
         "Price", "Seller Location", "Description", "Car Profile"]
    ].sort_values("Car Profile")
)

In [ ]:
categorical_columns = [
    "Make",
    "Model",
    "Fuel",
    "Registration city",
    "Assembly",
    "Transmission",
    "Condition",
    "Car documents"
]

for column in categorical_columns:
    print(f"\n{column}")
    print("Unique values:", df_clean[column].nunique())
    print(df_clean[column].value_counts(dropna=False).head(20))

In [ ]:
print("Possible Corolla spelling variations:")
display(
    df_clean[
        df_clean["Model"].str.contains("cor", case=False, na=False)
    ]["Model"].value_counts()
)

print("\nRare fuel records:")
display(
    df_clean[
        df_clean["Fuel"].isin(["Hybrid", "Diesel"])
    ][
        ["Ad ID", "Car Name", "Make", "Model", "Year",
         "KM's driven", "Price", "Fuel"]
    ]
)

print("\nRegistration categories:")
display(
    df_clean["Registration city"]
    .value_counts(dropna=False)
    .tail(25)
)

In [ ]:
df_clean["Model"] = df_clean["Model"].replace({
    "Corrolla Altis": "Corolla Altis"
})

df_clean["Registration city"] = df_clean["Registration city"].replace({
    "Sukkar": "Sukkur",
    "Toba Tek singh": "Toba Tek Singh"
})

print("Corrected Corolla models:")
print(df_clean[df_clean["Model"].str.contains(
    "cor", case=False, na=False
)]["Model"].value_counts())

print("\nCorrected registration values:")
print(
    df_clean["Registration city"].value_counts().head(20)
)

In [ ]:
print("All model names:")
for value in sorted(df_clean["Model"].dropna().unique()):
    print(value)

print("\nAll registration categories:")
for value in sorted(df_clean["Registration city"].dropna().unique()):
    print(value)

In [ ]:
df_clean["Registration city"] = df_clean["Registration city"].replace({
    "Sheikhüpura": "Sheikhupura"
})

In [ ]:
text_columns = [
    "Make", "Model", "Fuel", "Registration city",
    "Assembly", "Transmission", "Condition", "Car documents"
]

for column in text_columns:
    values = df_clean[column].dropna()

    leading_trailing_spaces = values[
        values != values.str.strip()
    ]

    print(f"\n{column}")
    print("Values with extra spaces:", len(leading_trailing_spaces))
    print("Missing values:", df_clean[column].isna().sum())

In [ ]:
print("Dataset shape:", df_clean.shape)
print("Exact duplicates:", df_clean.duplicated().sum())
print("Duplicate Ad IDs:", df_clean["Ad ID"].duplicated().sum())
print("Missing values:")
print(df_clean.isna().sum())

In [ ]:
output_path = PROJECT_ROOT / "data/processed/olx_cars_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("File:", output_path)
print("Shape:", df_clean.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
print("Price Statistics:")
display(df_clean["Price"].describe().to_frame())

plt.figure(figsize=(10, 6))
sns.histplot(df_clean["Price"], bins=50, kde=True)

plt.title("Distribution of Used Car Prices")
plt.xlabel("Price (PKR)")
plt.ylabel("Number of Listings")
plt.ticklabel_format(style="plain", axis="x")
plt.show()

In [ ]:
price_skewness = df_clean["Price"].skew()

print("Price Skewness:", price_skewness)

if price_skewness > 0:
    print("Distribution is positively (right) skewed.")
elif price_skewness < 0:
    print("Distribution is negatively (left) skewed.")
else:
    print("Distribution is approximately symmetric.")

In [ ]:
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df_clean,
    x="Year",
    y="Price",
    alpha=0.4
)

plt.title("Used Car Price vs Manufacturing Year")
plt.xlabel("Manufacturing Year")
plt.ylabel("Price (PKR)")
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
correlation = df_clean["Year"].corr(df_clean["Price"])

print("Year-Price Correlation:", correlation)

In [ ]:
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df_clean,
    x="KM's driven",
    y="Price",
    alpha=0.4
)

plt.title("Used Car Price vs Mileage")
plt.xlabel("Mileage (KM)")
plt.ylabel("Price (PKR)")
plt.ticklabel_format(style="plain", axis="both")
plt.show()

In [ ]:
correlation = df_clean["KM's driven"].corr(df_clean["Price"])

print("Mileage-Price Correlation:", correlation)

In [ ]:
plt.figure(figsize=(12, 7))

sns.boxplot(
    data=df_clean,
    x="Make",
    y="Price"
)

plt.title("Price Distribution by Car Manufacturer")
plt.xlabel("Car Manufacturer")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
brand_prices = (
    df_clean.groupby("Make")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

display(brand_prices)

In [ ]:
top_models = (
    df_clean["Model"]
    .value_counts()
    .head(15)
    .index
)

model_data = df_clean[
    df_clean["Model"].isin(top_models)
]

plt.figure(figsize=(14, 7))

sns.boxplot(
    data=model_data,
    x="Model",
    y="Price"
)

plt.title("Price Distribution of Top 15 Car Models")
plt.xlabel("Car Model")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
model_prices = (
    model_data.groupby("Model")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

display(model_prices)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(
    data=df_clean,
    x="Assembly",
    y="Price",
    ax=axes[0]
)

axes[0].set_title("Price by Assembly")
axes[0].ticklabel_format(style="plain", axis="y")

sns.boxplot(
    data=df_clean,
    x="Transmission",
    y="Price",
    ax=axes[1]
)

axes[1].set_title("Price by Transmission")
axes[1].ticklabel_format(style="plain", axis="y")

plt.tight_layout()
plt.show()

In [ ]:
assembly_prices = df_clean.groupby("Assembly")["Price"].agg(
    ["count", "mean", "median"]
)

transmission_prices = df_clean.groupby("Transmission")["Price"].agg(
    ["count", "mean", "median"]
)

print("Assembly:")
display(assembly_prices)

print("Transmission:")
display(transmission_prices)

In [ ]:
top_cities = (
    df_clean["Registration city"]
    .value_counts()
    .head(10)
    .index
)

city_data = df_clean[
    df_clean["Registration city"].isin(top_cities)
]

plt.figure(figsize=(14, 7))

sns.boxplot(
    data=city_data,
    x="Registration city",
    y="Price"
)

plt.title("Used Car Prices by Registration Category")
plt.xlabel("Registration Category")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
city_prices = (
    city_data.groupby("Registration city")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

display(city_prices)

In [ ]:
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df_clean,
    x="KM's driven",
    y="Price",
    hue="Year",
    palette="viridis",
    alpha=0.6,
    legend=False
)

plt.title("Price vs Mileage by Manufacturing Year")
plt.xlabel("Mileage (KM)")
plt.ylabel("Price (PKR)")
plt.ticklabel_format(style="plain", axis="both")
plt.show()

In [ ]:
mileage_counts = (
    df_clean["KM's driven"]
    .value_counts()
    .sort_values(ascending=False)
)

display(mileage_counts.head(20))

In [ ]:
low_mileage = df_clean[
    df_clean["KM's driven"] < 1000
]

display(
    low_mileage[
        ["Make", "Model", "Year", "KM's driven", "Price"]
    ]
)

In [ ]:
suspicious_mileage = df_clean[
    df_clean["KM's driven"].isin([1, 20, 12345, 123456, 500000])
]

display(
    suspicious_mileage[
        ["Make", "Model", "Year", "KM's driven", "Price", "Registration city"]
    ].sort_values("KM's driven")
)

In [ ]:
suspicious_values = [1, 20, 12345, 123456, 500000]

mileage_pattern_analysis = (
    df_clean[df_clean["KM's driven"].isin(suspicious_values)]
    .groupby("KM's driven")
    .agg(
        listings=("Price", "count"),
        average_price=("Price", "mean"),
        median_price=("Price", "median"),
        average_year=("Year", "mean")
    )
)

display(mileage_pattern_analysis)

In [ ]:
placeholder_mileage = df_clean[
    df_clean["KM's driven"].isin([12345, 123456])
]

placeholder_models = (
    placeholder_mileage
    .groupby(["Make", "Model", "KM's driven"])
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
)

display(placeholder_models.head(25))

In [ ]:
df_clean["Mileage_Suspected_Placeholder"] = (
    df_clean["KM's driven"].isin([1, 20, 12345, 123456])
).astype(int)

display(
    df_clean["Mileage_Suspected_Placeholder"].value_counts()
)

In [ ]:
df_clean["Car_Age"] = 2026 - df_clean["Year"]

display(
    df_clean["Car_Age"].describe()
)

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df_clean,
    x="Car_Age",
    bins=25,
    kde=True
)

plt.title("Distribution of Used Car Age")
plt.xlabel("Car Age (Years)")
plt.ylabel("Number of Listings")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df_clean,
    x="Car_Age",
    y="Price",
    alpha=0.5
)

plt.title("Car Age vs Used Car Price")
plt.xlabel("Car Age (Years)")
plt.ylabel("Price (PKR)")
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
age_price_corr = df_clean["Car_Age"].corr(df_clean["Price"])

print("Car Age vs Price Correlation:", age_price_corr)

In [ ]:
fuel_analysis = (
    df_clean.groupby("Fuel")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

display(fuel_analysis)

In [ ]:
documents_analysis = (
    df_clean.groupby("Car documents")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("median", ascending=False)
)

display(documents_analysis)

In [ ]:
seller_location_analysis = (
    df_clean.groupby("Seller Location")["Price"]
    .agg(["count", "mean", "median"])
    .sort_values("count", ascending=False)
)

display(seller_location_analysis.head(15))

In [ ]:
missing_analysis = (
    df_clean.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_analysis = missing_analysis[missing_analysis > 0]

display(missing_analysis)


In [ ]:
for column in df_clean.columns:
    print(f"{column}: {df_clean[column].nunique()} unique values")

In [ ]:
display(
    df_clean[
        ["Car Name", "Make", "Model", "Year", "Price"]
    ].sample(20, random_state=42)
)

In [ ]:
import re

cc_pattern = df_clean["Car Name"].str.extract(
    r'(\d{3,4})\s*cc',
    flags=re.IGNORECASE
)

print("Listings with engine capacity mentioned:",
      cc_pattern.notna().sum().iloc[0])

display(
    df_clean.loc[
        cc_pattern[0].notna(),
        ["Car Name", "Make", "Model", "Year", "Price"]
    ].head(20)
)

In [ ]:
display(
    df_clean["Car Features"].dropna().sample(
        15,
        random_state=42
    )
)

In [ ]:
from collections import Counter

feature_counts = Counter()

for features in df_clean["Car Features"].dropna():
    feature_list = [
        feature.strip()
        for feature in features.split(",")
    ]
    feature_counts.update(feature_list)

feature_frequency = pd.DataFrame(
    feature_counts.items(),
    columns=["Feature", "Listings"]
).sort_values("Listings", ascending=False)

display(feature_frequency)

In [ ]:
feature_series = df_clean["Car Features"].fillna("").str.strip()

feature_count_per_listing = feature_series.apply(
    lambda x: len([
        feature.strip()
        for feature in x.split(",")
        if feature.strip()
    ])
)

print("Empty feature listings:", feature_series.eq("").sum())
print("\nFeatures per listing:")
display(feature_count_per_listing.describe())

print("\nDistribution:")
display(feature_count_per_listing.value_counts().sort_index())

In [ ]:
display(
    df_clean["Description"].dropna().sample(
        15,
        random_state=42
    )
)

In [ ]:
print("Dataset shape:", df_clean.shape)

print("\nMissing values:")
display(
    df_clean.isnull().sum().sort_values(
        ascending=False
    )
)

print("\nData types:")
display(df_clean.dtypes)

In [ ]:
output_path = PROJECT_ROOT / "data/processed/olx_cars_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Location:", output_path)

In [ ]:
df = pd.read_csv(
    PROJECT_ROOT / "data/processed/olx_cars_cleaned.csv"
)

print("Shape:", df.shape)
print("Exact duplicates:", df.duplicated().sum())
print("Duplicate Ad IDs:", df["Ad ID"].duplicated().sum())

In [ ]:
all_features = sorted({
    feature.strip()
    for value in df["Car Features"].dropna()
    for feature in value.split(",")
    if feature.strip()
})

for feature in all_features:
    df[feature] = df["Car Features"].apply(
        lambda value: int(
            feature in [
                item.strip()
                for item in value.split(",")
            ]
        ) if pd.notna(value) else 0
    )

print("Extracted features:", len(all_features))
print("New dataset shape:", df.shape)

display(df[all_features].head())

In [ ]:
print("Unique values in extracted features:")

display(
    df[all_features].nunique().value_counts()
)

print("\nFeature frequency after extraction:")

display(
    df[all_features].sum()
    .sort_values(ascending=False)
    .rename("Listings")
    .to_frame()
)


In [ ]:
from datetime import datetime

current_year = datetime.now().year

df["Car_Age"] = current_year - df["Year"]

print("Current year:", current_year)
print("\nCar age statistics:")
display(df["Car_Age"].describe())

In [ ]:
df["KM_Per_Year"] = (
    df["KM's driven"] / df["Car_Age"].clip(lower=1)
)

print("KM per year statistics:")
display(df["KM_Per_Year"].describe())

print("\nHighest KM per year:")
display(
    df.nlargest(10, "KM_Per_Year")[
        ["Make", "Model", "Year", "KM's driven", "Car_Age", "KM_Per_Year"]
    ]
)

In [ ]:
print("Correlation with Price")

display(
    df[
        ["Price", "KM's driven", "Car_Age", "KM_Per_Year"]
    ].corr()["Price"]
)

In [ ]:
print(
    df["Registration city"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
province_names = ["Punjab", "Sindh"]

special_categories = ["Unregistered", "Unknown"]

def get_registration_type(value):
    if pd.isna(value):
        return "Missing"
    elif value in province_names:
        return "Province"
    elif value in special_categories:
        return value
    else:
        return "City"

df["Registration_Type"] = (
    df["Registration city"].apply(get_registration_type)
)

print("Registration type distribution:")
display(df["Registration_Type"].value_counts())

print("\nCross-check:")
display(
    pd.crosstab(
        df["Registration city"],
        df["Registration_Type"]
    )
)

In [ ]:
location_counts = df["Registration city"].value_counts()

df["Registration_Location_Group"] = df["Registration city"].apply(
    lambda value: (
        value
        if pd.isna(value)
        else value
        if value in ["Punjab", "Sindh", "Unregistered", "Unknown"]
        else value
        if location_counts.get(value, 0) >= 30
        else "Other"
    )
)

print("Grouped registration locations:")
display(
    df["Registration_Location_Group"]
    .value_counts(dropna=False)
)

In [ ]:
display(
    df["Seller Location"]
    .dropna()
    .sample(25, random_state=42)
)

In [ ]:
df["Seller_City"] = (
    df["Seller Location"]
    .str.rsplit(",", n=1)
    .str[-1]
    .str.strip()
)

print("Unique seller cities:", df["Seller_City"].nunique())

print("\nTop seller cities:")
display(df["Seller_City"].value_counts().head(20))

In [ ]:
no_comma = df[
    df["Seller Location"].notna()
    & ~df["Seller Location"].str.contains(",", regex=False)
]

print("Locations without a comma:", len(no_comma))

display(
    no_comma["Seller Location"]
    .value_counts()
    .head(25)
)

In [ ]:
df.loc[
    df["Seller_City"] == "Pakistan",
    "Seller_City"
] = pd.NA

print("Missing seller cities:", df["Seller_City"].isna().sum())

print("\nTop seller cities:")
display(df["Seller_City"].value_counts().head(15))

In [ ]:
province_names = ["Punjab", "Sindh", "Balochistan", "KPK"]

def get_seller_location_type(value):
    if pd.isna(value):
        return "Missing"
    elif value in province_names:
        return "Province"
    else:
        return "City"

df["Seller_Location_Type"] = (
    df["Seller_City"].apply(get_seller_location_type)
)

display(
    df["Seller_Location_Type"]
    .value_counts(dropna=False)
)


In [ ]:
engineered_columns = [
    "Car_Age",
    "KM_Per_Year",
    "Mileage_Suspected_Placeholder",
    "Registration_Type",
    "Registration_Location_Group",
    "Seller_City",
    "Seller_Location_Type"
]

display(df[engineered_columns].head())

print("Dataset shape:", df.shape)
print("\nMissing values in engineered features:")
display(df[engineered_columns].isna().sum())

In [ ]:
print(
    df[
        [
            "Car_Age",
            "KM_Per_Year",
            "Mileage_Suspected_Placeholder",
            "Registration_Type",
            "Registration_Location_Group",
            "Seller_City",
            "Seller_Location_Type"
        ]
    ].head(10).to_string()
)

In [ ]:
numeric_columns = [
    "Price",
    "Year",
    "KM's driven",
    "Car_Age",
    "KM_Per_Year"
]

display(df[numeric_columns].describe().round(2))

print("\nInvalid or non-positive values:")
display(
    (df[numeric_columns] <= 0).sum()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Price"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Original Price Distribution")
axes[0].set_xlabel("Price (PKR)")

sns.histplot(
    np.log1p(df["Price"]),
    bins=40,
    kde=True,
    ax=axes[1]
)
axes[1].set_title("Log-Transformed Price Distribution")
axes[1].set_xlabel("Log(Price + 1)")

plt.tight_layout()
plt.show()

In [ ]:
for i, column in enumerate(df.columns, start=1):
    print(i, column)

In [ ]:
car_feature_columns = all_features

categorical_features = [
    "Make",
    "Model",
    "Fuel",
    "Car documents",
    "Assembly",
    "Transmission",
    "Registration_Type",
    "Registration_Location_Group",
    "Seller_City",
    "Seller_Location_Type"
]

numerical_features = [
    "Year",
    "KM's driven",
    "KM_Per_Year",
    "Mileage_Suspected_Placeholder"
]

feature_columns = (
    categorical_features
    + numerical_features
    + car_feature_columns
)

X = df[feature_columns].copy()
y = df["Price"].copy()

print("Number of features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

In [ ]:
print("Categorical features:")
display(X[categorical_features].dtypes)

print("\nNumerical features:")
display(X[numerical_features].dtypes)

print("\nBinary car features:")
display(X[car_feature_columns].dtypes.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining percentage:", round(len(X_train) / len(X) * 100, 2), "%")
print("Testing percentage:", round(len(X_test) / len(X) * 100, 2), "%")

print("\nTraining price mean:", round(y_train.mean()))
print("Testing price mean:", round(y_test.mean()))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features),
    ("binary", "passthrough", car_feature_columns)
])

print("Preprocessing pipeline created successfully.")

In [ ]:
X_train[categorical_features] = (
    X_train[categorical_features].replace({pd.NA: np.nan})
)

X_test[categorical_features] = (
    X_test[categorical_features].replace({pd.NA: np.nan})
)

print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

print("\nMissing categorical values:")
display(X_train[categorical_features].isna().sum())

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(X_train_processed, y_train)

print("Linear Regression trained successfully.")

In [ ]:
y_pred = baseline_model.predict(X_test_processed)

print("Actual prices:")
print(y_test.head(10).values)

print("\nPredicted prices:")
print(y_pred[:10].round(0))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: Rs. {mae:,.0f}")
print(f"RMSE: Rs. {rmse:,.0f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
from sklearn.dummy import DummyRegressor

dummy_model = DummyRegressor(strategy="median")

dummy_model.fit(X_train_processed, y_train)

dummy_pred = dummy_model.predict(X_test_processed)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))
dummy_r2 = r2_score(y_test, dummy_pred)

print(f"Dummy MAE: Rs. {dummy_mae:,.0f}")
print(f"Dummy RMSE: Rs. {dummy_rmse:,.0f}")
print(f"Dummy R²: {dummy_r2:.4f}")

In [ ]:
mae_improvement = (
    (dummy_mae - mae) / dummy_mae
) * 100

rmse_improvement = (
    (dummy_rmse - rmse) / dummy_rmse
) * 100

print(f"MAE improvement: {mae_improvement:.2f}%")
print(f"RMSE improvement: {rmse_improvement:.2f}%")

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

tree_model.fit(X_train_processed, y_train)

print("Decision Tree trained successfully.")

In [ ]:
tree_pred = tree_model.predict(X_test_processed)

tree_mae = mean_absolute_error(y_test, tree_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
tree_r2 = r2_score(y_test, tree_pred)

print(f"Decision Tree MAE: Rs. {tree_mae:,.0f}")
print(f"Decision Tree RMSE: Rs. {tree_rmse:,.0f}")
print(f"Decision Tree R²: {tree_r2:.4f}")

In [ ]:
tree_train_pred = tree_model.predict(X_train_processed)

train_mae = mean_absolute_error(y_train, tree_train_pred)
train_r2 = r2_score(y_train, tree_train_pred)

print(f"Training MAE: Rs. {train_mae:,.0f}")
print(f"Testing MAE: Rs. {tree_mae:,.0f}")

print(f"\nTraining R²: {train_r2:.4f}")
print(f"Testing R²: {tree_r2:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully.")

In [ ]:
rf_pred = rf_model.predict(X_test_processed)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f"Random Forest MAE: Rs. {rf_mae:,.0f}")
print(f"Random Forest RMSE: Rs. {rf_rmse:,.0f}")
print(f"Random Forest R²: {rf_r2:.4f}")

In [ ]:
rf_train_pred = rf_model.predict(X_train_processed)

rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_train_r2 = r2_score(y_train, rf_train_pred)

print(f"Training MAE: Rs. {rf_train_mae:,.0f}")
print(f"Testing MAE: Rs. {rf_mae:,.0f}")

print(f"\nTraining R²: {rf_train_r2:.4f}")
print(f"Testing R²: {rf_r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Dummy",
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        dummy_mae,
        mae,
        tree_mae,
        rf_mae
    ],
    "RMSE": [
        dummy_rmse,
        rmse,
        tree_rmse,
        rf_rmse
    ],
    "R2": [
        dummy_r2,
        r2,
        tree_r2,
        rf_r2
    ]
})

results = results.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

results

In [ ]:
%pip install xgboost

In [ ]:
import xgboost

print("XGBoost version:", xgboost.__version__)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost trained successfully.")

In [ ]:
xgb_pred = xgb_model.predict(X_test_processed)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print(f"XGBoost MAE: Rs. {xgb_mae:,.0f}")
print(f"XGBoost RMSE: Rs. {xgb_rmse:,.0f}")
print(f"XGBoost R²: {xgb_r2:.4f}")

In [ ]:
xgb_train_pred = xgb_model.predict(X_train_processed)

xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_train_r2 = r2_score(y_train, xgb_train_pred)

print(f"Training MAE: Rs. {xgb_train_mae:,.0f}")
print(f"Testing MAE: Rs. {xgb_mae:,.0f}")

print(f"\nTraining R²: {xgb_train_r2:.4f}")
print(f"Testing R²: {xgb_r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Dummy",
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        dummy_mae,
        mae,
        tree_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        dummy_rmse,
        rmse,
        tree_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2": [
        dummy_r2,
        r2,
        tree_r2,
        rf_r2,
        xgb_r2
    ]
})

results = results.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

results

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    xgb_model,
    X_train_processed,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=1
)

print("CV MAE:", -cv_results["test_mae"].mean())
print("CV RMSE:", -cv_results["test_rmse"].mean())
print("CV R²:", cv_results["test_r2"].mean())

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance.head(20)

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(20).sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 20 Features Influencing CarQeemat Predictions")

plt.tight_layout()
plt.show()

In [ ]:
%pip install shap

In [ ]:
import shap

print("SHAP version:", shap.__version__)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)

print("SHAP explainer created successfully.")

In [ ]:
X_shap = X_test_processed[:300]

shap_values = explainer.shap_values(X_shap)

print("SHAP values shape:", shap_values.shape)

In [ ]:
X_shap_dense = X_shap.toarray()

shap.summary_plot(
    shap_values,
    X_shap_dense,
    feature_names=feature_names,
    max_display=20
)

In [ ]:
car_index = 0

print("Actual price:", y_test.iloc[car_index])
print("Predicted price:", rf"{xgb_pred[car_index]:,.0f}")

In [ ]:
car_index = 0

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[car_index],
        base_values=explainer.expected_value,
        data=X_shap_dense[car_index],
        feature_names=feature_names
    ),
    max_display=15
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

print("Hyperparameter tuning tools imported successfully.")


In [ ]:
param_dist = {
    "n_estimators": randint(300, 1001),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(0.02, 0.08),
    "min_child_weight": randint(3, 11),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(1, 9)
}

print("Search space defined.")
print("Parameters:", list(param_dist.keys()))

In [ ]:
tuning_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=tuning_cv,
    verbose=1,
    random_state=42,
    n_jobs=1,
    refit=True
)

print("RandomizedSearchCV configured successfully.")

In [ ]:
random_search.fit(X_train_processed, y_train)

print("Hyperparameter tuning completed.")

In [ ]:
print("Best CV MAE:", -random_search.best_score_)
print("\nBest hyperparameters:")
for parameter, value in random_search.best_params_.items():
    print(f"{parameter}: {value}")

In [ ]:
best_xgb_model = random_search.best_estimator_

print("Best XGBoost model retrieved.")

In [ ]:
best_xgb_pred = best_xgb_model.predict(X_test_processed)

tuned_mae = mean_absolute_error(y_test, best_xgb_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, best_xgb_pred))
tuned_r2 = r2_score(y_test, best_xgb_pred)

print(f"Tuned MAE: Rs {tuned_mae:,.2f}")
print(f"Tuned RMSE: Rs {tuned_rmse:,.2f}")
print(f"Tuned R2: {tuned_r2:.4f}")

In [ ]:
final_model = xgb_model
final_predictions = xgb_pred

print("Original XGBoost selected as CarQeemat's champion model.")

In [ ]:
y_train_log = np.log1p(y_train)

print("Original target range:")
print(y_train.min(), "to", y_train.max())

print("\nLog-transformed target range:")
print(y_train_log.min(), "to", y_train_log.max())

In [ ]:
xgb_log_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_log_model.fit(X_train_processed, y_train_log)

print("Log-target XGBoost training completed.")

In [ ]:
log_predictions = xgb_log_model.predict(X_test_processed)

log_predictions_pkr = np.expm1(log_predictions)

print("First 10 predictions in PKR:")
print(np.round(log_predictions_pkr[:10]))

In [ ]:
log_mae = mean_absolute_error(y_test, log_predictions_pkr)
log_rmse = np.sqrt(mean_squared_error(y_test, log_predictions_pkr))
log_r2 = r2_score(y_test, log_predictions_pkr)

print(f"Log Model MAE: Rs {log_mae:,.2f}")
print(f"Log Model RMSE: Rs {log_rmse:,.2f}")
print(f"Log Model R2: {log_r2:.4f}")

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Original XGBoost",
        "Tuned XGBoost",
        "Log-target XGBoost"
    ],
    "MAE": [
        276582.28,
        208565.09,
        174805.47,
        158226.11,
        tuned_mae,
        log_mae
    ],
    "RMSE": [
        374122,
        295353,
        256909,
        232512,
        tuned_rmse,
        log_rmse
    ],
    "R2": [
        0.8935,
        0.9336,
        0.9498,
        0.9589,
        tuned_r2,
        log_r2
    ]
})

final_comparison = final_comparison.sort_values(
    by="MAE"
).reset_index(drop=True)

final_comparison

In [ ]:
prediction_results = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": final_predictions
})

prediction_results["Error"] = (
    prediction_results["Actual_Price"]
    - prediction_results["Predicted_Price"]
)

prediction_results["Absolute_Error"] = (
    prediction_results["Error"].abs()
)

prediction_results.head(10)

In [ ]:
prediction_results["Price_Range"] = pd.cut(
    prediction_results["Actual_Price"],
    bins=[0, 1000000, 2000000, 3000000, 4000000, 5000001],
    labels=[
        "Below 10 Lakh",
        "10-20 Lakh",
        "20-30 Lakh",
        "30-40 Lakh",
        "40-50 Lakh"
    ]
)

price_range_analysis = prediction_results.groupby(
    "Price_Range",
    observed=False
).agg(
    Listings=("Actual_Price", "count"),
    MAE=("Absolute_Error", "mean"),
    Average_Error=("Error", "mean")
).round(2)

price_range_analysis

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    price_range_analysis.index.astype(str),
    price_range_analysis["MAE"]
)

plt.xlabel("Actual Car Price Range")
plt.ylabel("Mean Absolute Error (PKR)")
plt.title("CarQeemat Prediction Error Across Price Ranges")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
prediction_results["Make"] = df.loc[
    X_test.index, "Make"
].values

print("Manufacturer information added.")
print("Unique manufacturers:", prediction_results["Make"].nunique())

prediction_results[["Make", "Actual_Price", "Predicted_Price"]].head(10)

In [ ]:
manufacturer_analysis = prediction_results.groupby("Make").agg(
    Listings=("Actual_Price", "count"),
    MAE=("Absolute_Error", "mean"),
    Average_Error=("Error", "mean")
).sort_values(
    by="MAE"
).round(2)

manufacturer_analysis

In [ ]:
manufacturer_plot = manufacturer_analysis.sort_values(
    by="MAE",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    manufacturer_plot.index,
    manufacturer_plot["MAE"]
)

plt.xlabel("Mean Absolute Error (PKR)")
plt.ylabel("Manufacturer")
plt.title("CarQeemat Prediction Error by Manufacturer")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    prediction_results["Actual_Price"],
    prediction_results["Predicted_Price"],
    alpha=0.5
)

plt.plot(
    [0, 5000000],
    [0, 5000000],
    linestyle="--"
)

plt.xlabel("Actual Price (PKR)")
plt.ylabel("Predicted Price (PKR)")
plt.title("Actual vs Predicted Car Prices")
plt.tight_layout()
plt.show()

In [ ]:
prediction_results.nlargest(10, "Absolute_Error")[
    [
        "Make",
        "Actual_Price",
        "Predicted_Price",
        "Error",
        "Absolute_Error",
        "Price_Range"
    ]
]

In [ ]:
worst_indices = prediction_results.nlargest(
    10, "Absolute_Error"
).index

df.loc[
    worst_indices,
    [
        "Car Name",
        "Make",
        "Model",
        "Year",
        "KM's driven",
        "Price",
        "Fuel",
        "Assembly",
        "Transmission",
        "Car documents",
        "Registration city",
        "Seller Location"
    ]
]

In [ ]:
print(df[["Car Name", "Make", "Model", "Registration city"]].head(10))

In [ ]:
worst_indices = prediction_results.nlargest(
    10, "Absolute_Error"
).index

df.loc[
    worst_indices,
    [
        "Year",
        "Car_Age",
        "KM's driven",
        "KM_Per_Year",
        "Mileage_Suspected_Placeholder",
        "Registration_Type",
        "Registration_Location_Group",
        "Seller_City"
    ]
]

In [ ]:
df.loc[worst_indices]

In [ ]:
worst_positions = (
    prediction_results["Absolute_Error"]
    .to_numpy()
    .argsort()[-10:][::-1]
)

source_indices = X_test.iloc[worst_positions].index

worst_details = df.loc[
    source_indices,
    ["Ad ID", "Car Name", "Make", "Model", "Year", "KM's driven", "Price"]
].copy()

worst_details.insert(
    0, "Prediction_Row", worst_positions
)

worst_details["Model_Prediction"] = final_predictions[worst_positions]

worst_details["Price_Matches"] = (
    worst_details["Price"].to_numpy()
    == y_test.iloc[worst_positions].to_numpy()
)

worst_details

In [ ]:
name_year = df["Car Name"].str.extract(
    r"\b((?:19|20)\d{2})\b"
)[0]

year_check = pd.DataFrame({
    "Car Name": df["Car Name"],
    "Structured_Year": df["Year"],
    "Year_In_Name": pd.to_numeric(name_year, errors="coerce")
})

year_mismatches = year_check[
    year_check["Year_In_Name"].notna()
    & (year_check["Structured_Year"] != year_check["Year_In_Name"])
]

print("Potential year inconsistencies:", len(year_mismatches))

year_mismatches.head(15)

In [ ]:
year_mismatch_indices = year_mismatches.index

test_mismatches = len(
    year_mismatch_indices.intersection(X_test.index)
)

train_mismatches = len(
    year_mismatch_indices.intersection(X_train.index)
)

print("Training mismatches:", train_mismatches)
print("Testing mismatches:", test_mismatches)
print("Total mismatches:", len(year_mismatch_indices))

In [ ]:
test_year_mismatch = prediction_results.index.isin(
    year_mismatches.index
)

year_error_analysis = pd.DataFrame({
    "Year_Consistency": [
        "Consistent",
        "Potentially Inconsistent"
    ],
    "Listings": [
        (~test_year_mismatch).sum(),
        test_year_mismatch.sum()
    ],
    "MAE": [
        prediction_results.loc[
            ~test_year_mismatch, "Absolute_Error"
        ].mean(),
        prediction_results.loc[
            test_year_mismatch, "Absolute_Error"
        ].mean()
    ]
})

year_error_analysis.round(2)


In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    prediction_results["Error"],
    bins=40
)

plt.axvline(0, linestyle="--")

plt.xlabel("Prediction Error (PKR)")
plt.ylabel("Number of Cars")
plt.title("Distribution of CarQeemat Prediction Errors")

plt.tight_layout()
plt.show()

In [ ]:
residual_analysis = prediction_results["Error"].describe()

print(residual_analysis)

print(
    "\nMean Prediction Error:",
    round(prediction_results["Error"].mean(), 2)
)

print(
    "Median Prediction Error:",
    round(prediction_results["Error"].median(), 2)
)

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    prediction_results["Actual_Price"],
    prediction_results["Error"],
    alpha=0.5
)

plt.axhline(0, linestyle="--")

plt.xlabel("Actual Car Price (PKR)")
plt.ylabel("Prediction Error (PKR)")
plt.title("Prediction Errors vs Actual Car Prices")

plt.tight_layout()
plt.show()

In [ ]:
prediction_results["Percentage_Error"] = (
    prediction_results["Error"]
    / prediction_results["Actual_Price"]
) * 100

prediction_results["Absolute_Percentage_Error"] = (
    prediction_results["Percentage_Error"].abs()
)

print(
    "Mean Absolute Percentage Error:",
    round(
        prediction_results["Absolute_Percentage_Error"].mean(),
        2
    ),
    "%"
)

print(
    "Median Absolute Percentage Error:",
    round(
        prediction_results["Absolute_Percentage_Error"].median(),
        2
    ),
    "%"
)

In [ ]:
percentage_error_analysis = prediction_results.groupby(
    "Price_Range",
    observed=False
).agg(
    Listings=("Actual_Price", "count"),
    MAPE=("Absolute_Percentage_Error", "mean"),
    Median_Percentage_Error=("Absolute_Percentage_Error", "median")
).round(2)

percentage_error_analysis

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    percentage_error_analysis.index.astype(str),
    percentage_error_analysis["MAPE"]
)

plt.xlabel("Actual Car Price Range")
plt.ylabel("Mean Absolute Percentage Error (%)")
plt.title("CarQeemat Relative Prediction Error by Price Range")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

carqeemat_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

pipeline_cv_results = cross_validate(
    carqeemat_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=1
)

print("Pipeline CV MAE:", round(-pipeline_cv_results["test_mae"].mean(), 2))
print("Pipeline CV RMSE:", round(-pipeline_cv_results["test_rmse"].mean(), 2))
print("Pipeline CV R2:", round(pipeline_cv_results["test_r2"].mean(), 4))

In [ ]:
carqeemat_pipeline.fit(X_train, y_train)

pipeline_predictions = carqeemat_pipeline.predict(X_test)

print("First 10 actual prices:")
print(y_test.iloc[:10].values)

print("\nFirst 10 pipeline predictions:")
print(pipeline_predictions[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

pipeline_mae = mean_absolute_error(y_test, pipeline_predictions)

pipeline_rmse = np.sqrt(
    mean_squared_error(y_test, pipeline_predictions)
)

pipeline_r2 = r2_score(y_test, pipeline_predictions)

print("Pipeline Test MAE:", round(pipeline_mae, 2))
print("Pipeline Test RMSE:", round(pipeline_rmse, 2))
print("Pipeline Test R2:", round(pipeline_r2, 4))

In [ ]:
import joblib
from pathlib import Path

model_path = Path(
    PROJECT_ROOT / "models/carqeemat_pipeline.pkl"
)

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(carqeemat_pipeline, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())
print("File size (MB):", round(model_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
loaded_model = joblib.load(
    PROJECT_ROOT / "models/carqeemat_pipeline.pkl"
)

print("Saved model loaded successfully!")
print("Model type:", type(loaded_model))

In [ ]:
sample_car = X_test.iloc[[0]]

saved_prediction = loaded_model.predict(sample_car)[0]

actual_price = y_test.iloc[0]

print("Actual Price: Rs", round(actual_price))
print("Predicted Price: Rs", round(saved_prediction))
print("Absolute Error: Rs", round(abs(actual_price - saved_prediction)))